We start by making a connection with Airport database.

In [2]:
import mysql.connector
from mysql.connector import Error
import prettytable

#Creating a connection with Employees database

def create_connection(host_name, user_name, user_password):
    connection = None
    try:
        connection = mysql.connector.connect(
            host=host_name,
            user=user_name,
            passwd=user_password,
            database='airportdb'
        )
        print("Connection to MySQL DB successful")
    except Error as e:
        print(f"The error '{e}' occurred")

    return connection

connection = create_connection("localhost", "root", "kurssql11")
my_cursor=connection.cursor()  


Connection to MySQL DB successful


We find how long on average are flights chosen by man and women.

In [2]:
my_cursor.execute("""
WITH duration AS
			(
			SELECT
				p.passenger_id,
				p.sex,
				timestampdiff(minute, f.departure, f.arrival) AS minutes
			FROM booking b
			JOIN flight f
				ON f.flight_id=b.flight_id
			JOIN passengerdetails p
				ON p.passenger_id=b.passenger_id
			),
            
avg_duration AS
			(
			SELECT
				sex,
				avg(minutes) as average_duration
			FROM duration
			GROUP BY sex
			)
            
            
SELECT
	sex,
    CONCAT(FLOOR(average_duration/60),'h',FLOOR(MOD(average_duration,60)),'m') as average_time
FROM avg_duration
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+-----+--------------+
| sex | average_time |
+-----+--------------+
|  m  |    10h34m    |
|  w  |    10h34m    |
+-----+--------------+


As this query took long time to fetch (213sec) I decided to optimise it.
First observation is that in the previous aproach we computed timestampdiff for each booking but in fact we can compute it for each flight instead.

In [3]:
my_cursor.execute("""

WITH duration AS
			 (
			 SELECT
				flight_id,
				timestampdiff(minute, departure, arrival) AS minutes
			 from flight
			),
            
            
sex_flight  AS
			(
			SELECT
				b.flight_id,
				p.sex
			FROM booking b
			JOIN passengerdetails p
				ON p.passenger_id=b.passenger_id
			),
            
avg_duration AS 
		(
		SELECT
			s.sex,
			avg(minutes) as average_time
		FROM sex_flight s
		JOIN duration d
			ON d.flight_id=s.flight_id
		GROUP BY sex
		)
        
SELECT 
	sex,
    CONCAT(FLOOR(average_time/60),'h',FLOOR(MOD(average_time,60)),'m') as average_time
FROM 
	avg_duration
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)


+-----+--------------+
| sex | average_time |
+-----+--------------+
|  m  |    10h34m    |
|  w  |    10h34m    |
+-----+--------------+


We've managed to reduce the fetching time to 194sec (so aprox. by 9%).

Next we find passengers that took the most flights.

In [2]:
my_cursor.execute("""
SELECT 
	p.firstname as first_name,
    p.lastname as last_name,
    COUNT(*) AS number_of_flights
FROM booking b
LEFT JOIN passenger p
	ON p.passenger_id=b.passenger_id
GROUP BY b.passenger_id
ORDER BY number_of_flights DESC
LIMIT 10;
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)


+------------+------------+-------------------+
| first_name | last_name  | number_of_flights |
+------------+------------+-------------------+
|   Khalid   |   Reeves   |        1945       |
|    Bart    |  Johnson   |        1902       |
|  Anthony   |  DiCosmo   |        1884       |
|    Josh    |  Saviano   |        1869       |
|    Tony    | DeGregorio |        1862       |
|            |  Steve-O   |        1846       |
|   Robert   |   Daniel   |        1845       |
|  Juaquin   |  Hawkins   |        1844       |
|   Keith    |   Poole    |        1837       |
|  Correll   | Buckhalter |        1836       |
+------------+------------+-------------------+


The average, min and max number of flights are:

In [4]:
my_cursor.execute("""
WITH passengers_flights as 
			(
			SELECT 
				passenger_id,
				COUNT(*) AS number_of_flights
			FROM booking 
			GROUP BY passenger_id
			)
            
SELECT 
	FLOOR(AVG(number_of_flights)) as average_number_of_flights,
    MIN(number_of_flights) as minimal_number_of_flights,
    MAX(number_of_flights) as maximal_number_of_flights
FROM passengers_flights;
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+---------------------------+---------------------------+---------------------------+
| average_number_of_flights | minimal_number_of_flights | maximal_number_of_flights |
+---------------------------+---------------------------+---------------------------+
|            1504           |            1152           |            1945           |
+---------------------------+---------------------------+---------------------------+


Most frequent travelers:

In [3]:
my_cursor.execute("""
WITH ranking AS
			(
			SELECT
				passenger_id,
				count(*) AS number_of_flights
			FROM booking
			GROUP BY passenger_id
			ORDER BY number_of_flights DESC 
			LIMIT 10
			)
            
SELECT
	p.firstname,
    p.lastname,
    r.number_of_flights
FROM ranking r
JOIN passenger p
	ON r.passenger_id=p.passenger_id
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+-----------+------------+-------------------+
| firstname |  lastname  | number_of_flights |
+-----------+------------+-------------------+
|   Khalid  |   Reeves   |        1945       |
|    Bart   |  Johnson   |        1902       |
|  Anthony  |  DiCosmo   |        1884       |
|    Josh   |  Saviano   |        1869       |
|    Tony   | DeGregorio |        1862       |
|           |  Steve-O   |        1846       |
|   Robert  |   Daniel   |        1845       |
|  Juaquin  |  Hawkins   |        1844       |
|   Keith   |   Poole    |        1837       |
|  Correll  | Buckhalter |        1836       |
+-----------+------------+-------------------+


Now we find passengers travelling to the largest number of countries.

In [4]:
my_cursor.execute("""
WITH ranking AS
			(
			SELECT
				b.passenger_id,
				COUNT(ag.country) AS number_of_countries
			FROM booking b
			JOIN flight f
				ON b.flight_id=f.flight_id
			JOIN airport_geo ag
				ON ag.airport_id=f.to
			GROUP BY b.passenger_id
            ORDER BY number_of_countries DESC
            LIMIT 10
			)
            
SELECT
	p.firstname,
    p.lastname,
	r.number_of_countries
FROM ranking r
JOIN passenger p
	ON r.passenger_id=p.passenger_id
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+-----------+------------+---------------------+
| firstname |  lastname  | number_of_countries |
+-----------+------------+---------------------+
|   Khalid  |   Reeves   |         1945        |
|    Bart   |  Johnson   |         1902        |
|  Anthony  |  DiCosmo   |         1884        |
|    Josh   |  Saviano   |         1869        |
|    Tony   | DeGregorio |         1862        |
|           |  Steve-O   |         1846        |
|   Robert  |   Daniel   |         1845        |
|  Juaquin  |  Hawkins   |         1844        |
|   Keith   |   Poole    |         1837        |
|  Correll  | Buckhalter |         1836        |
+-----------+------------+---------------------+
